LOAD DATASET & STUDY RAINFALL

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

df = pd.read_csv("../datasets/processed/climate_anomalies.csv")

print("Dataset Loaded Successfully")
print("Shape:", df.shape)

print("\nPRECIP_MM SUMMARY")
print(df["precip_mm"].describe())

print("\nTOP 20 RAINFALL VALUES")
print(df["precip_mm"].value_counts().head(20))

print("\nPERCENTAGE OF ZERO RAINFALL")
zero_rain = (df["precip_mm"] == 0).mean() * 100
print(f"{zero_rain:.2f}%")

Dataset Loaded Successfully
Shape: (119398, 149)

PRECIP_MM SUMMARY
count    119398.000000
mean          0.084331
std           0.608602
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max          43.800000
Name: precip_mm, dtype: float64

TOP 20 RAINFALL VALUES
precip_mm
0.00    104510
0.01      1551
0.10       900
0.02       862
0.03       585
0.20       476
0.04       469
0.05       414
0.06       333
0.30       300
0.08       291
0.07       287
0.40       268
0.50       228
0.09       227
0.12       189
0.60       186
0.70       170
0.13       160
0.11       160
Name: count, dtype: int64

PERCENTAGE OF ZERO RAINFALL
87.53%


RAINFALL BUCKET ANALYSIS

In [2]:
import pandas as pd

bins = [-1, 0, 1, float("inf")]

labels = ["Low", "Medium", "High"]

df["rainfall_risk"] = pd.cut(df["precip_mm"], bins=bins, labels=labels)

print(df["rainfall_risk"].value_counts())

print("\nPercentage Distribution")

print(round(df["rainfall_risk"].value_counts(normalize=True) * 100, 2))

rainfall_risk
Low       104510
Medium     12184
High        2704
Name: count, dtype: int64

Percentage Distribution
rainfall_risk
Low       87.53
Medium    10.20
High       2.26
Name: proportion, dtype: float64


C:\Users\ANAMIKA\AppData\Local\Temp\ipykernel_19056\1244748928.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["rainfall_risk"] = pd.cut(df["precip_mm"], bins=bins, labels=labels)


REMOVE LEAKAGE FEATURES

In [3]:
remove_cols = [
    "precip_mm",
    "climate_cluster",
    "climate_profile",
    "anomaly_flag",
    "anomaly_status",
]

X = df.drop(columns=remove_cols + ["rainfall_risk"])

y = df["rainfall_risk"]

print("Feature Matrix Shape:", X.shape)
print("Target Shape:", y.shape)

print("\nTarget Distribution")
print(y.value_counts())

Feature Matrix Shape: (119398, 144)
Target Shape: (119398,)

Target Distribution
rainfall_risk
Low       104510
Medium     12184
High        2704
Name: count, dtype: int64


TRAIN TEST SPLIT

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("X Train:", X_train.shape)
print("X Test :", X_test.shape)

print("\nTrain Distribution")
print(y_train.value_counts(normalize=True) * 100)

print("\nTest Distribution")
print(y_test.value_counts(normalize=True) * 100)

X Train: (95518, 144)
X Test : (23880, 144)

Train Distribution
rainfall_risk
Low       87.531146
Medium    10.204359
High       2.264495
Name: proportion, dtype: float64

Test Distribution
rainfall_risk
Low       87.529313
Medium    10.205193
High       2.265494
Name: proportion, dtype: float64


LABEL ENCODING

In [5]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

print("Classes:")
print(label_encoder.classes_)

Classes:
['High' 'Low' 'Medium']


LOGISTIC REGRESSION BASELINE

In [6]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(max_iter=5000, random_state=42, class_weight="balanced")

log_reg.fit(X_train, y_train_encoded)

print("Logistic Regression Trained Successfully")

Logistic Regression Trained Successfully


c:\Projects\ClimateGuardAI\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 5000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=5000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


PREDICTIONS

In [7]:
y_pred = log_reg.predict(X_test)

print("Predictions Generated")

Predictions Generated


MODEL EVALUATION

In [8]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

accuracy = accuracy_score(y_test_encoded, y_pred)

print(f"Accuracy: {accuracy:.4f}")

print("\nClassification Report")
print(
    classification_report(y_test_encoded, y_pred, target_names=label_encoder.classes_)
)

print("\nConfusion Matrix")
print(confusion_matrix(y_test_encoded, y_pred))

Accuracy: 0.8801

Classification Report
              precision    recall  f1-score   support

        High       0.25      0.77      0.38       541
         Low       0.99      0.90      0.95     20902
      Medium       0.54      0.71      0.61      2437

    accuracy                           0.88     23880
   macro avg       0.59      0.80      0.65     23880
weighted avg       0.93      0.88      0.90     23880


Confusion Matrix
[[  419     6   116]
 [  659 18867  1376]
 [  592   115  1730]]


FEATURE IMPORTANCE CHECK

In [9]:
suspicious_columns = [
    col for col in X.columns if "rain" in col.lower() or "precip" in col.lower()
]

print("Potential Leakage Features")
print(suspicious_columns)

Potential Leakage Features
[]


FINAL MODEL DATASET

In [10]:
remove_cols = [
    "precip_mm",
    "climate_cluster",
    "climate_profile",
    "anomaly_flag",
    "anomaly_status",
    "anomaly_score",
]

X = df.drop(columns=remove_cols + ["rainfall_risk"])

y = df["rainfall_risk"]

print("Final Feature Matrix")
print(X.shape)

print("\nTarget")
print(y.shape)

Final Feature Matrix
(119398, 143)

Target
(119398,)


RANDOM FOREST BASELINE

In [11]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)

rf_model.fit(X_train, y_train)

print("Random Forest Trained Successfully")

Random Forest Trained Successfully


RANDOM FOREST PREDICTIONS

In [12]:
y_pred_rf = rf_model.predict(X_test)

print("Predictions Generated Successfully")

Predictions Generated Successfully


RANDOM FOREST EVALUATION

In [13]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

accuracy = accuracy_score(y_test, y_pred_rf)

print("\nAccuracy:")
print(round(accuracy, 4))

print("\nClassification Report")
print(classification_report(y_test, y_pred_rf))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred_rf))


Accuracy:
0.9679

Classification Report
              precision    recall  f1-score   support

        High       0.84      0.78      0.81       541
         Low       1.00      0.98      0.99     20902
      Medium       0.79      0.95      0.86      2437

    accuracy                           0.97     23880
   macro avg       0.88      0.90      0.89     23880
weighted avg       0.97      0.97      0.97     23880


Confusion Matrix
[[  421     1   119]
 [   12 20380   510]
 [   66    59  2312]]


In [14]:
print("X Columns:", len(X.columns))
print("RF Importances:", len(rf_model.feature_importances_))

X Columns: 143
RF Importances: 144


In [15]:
from sklearn.metrics import accuracy_score

train_pred = rf_model.predict(X_train)

train_accuracy = accuracy_score(y_train, train_pred)

print("Train Accuracy:", round(train_accuracy, 4))

Train Accuracy: 0.9871


Retrain Random Forest On Final Features

In [16]:
print(X.shape)
print(y.shape)

(119398, 143)
(119398,)


In [17]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(X_train.shape)
print(X_test.shape)

(95518, 143)
(23880, 143)


In [18]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
)

rf_model.fit(X_train, y_train)

print("Random Forest Retrained Successfully")

Random Forest Retrained Successfully


Re-Evaluate Retrained Model

In [19]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred_rf = rf_model.predict(X_test)

print("Accuracy:")
print(round(accuracy_score(y_test, y_pred_rf), 4))

print("\nClassification Report")
print(classification_report(y_test, y_pred_rf))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred_rf))

Accuracy:
0.9616

Classification Report
              precision    recall  f1-score   support

        High       0.73      0.64      0.69       541
         Low       1.00      0.97      0.98     20902
      Medium       0.76      0.94      0.84      2437

    accuracy                           0.96     23880
   macro avg       0.83      0.85      0.84     23880
weighted avg       0.97      0.96      0.96     23880


Confusion Matrix
[[  348     1   192]
 [   31 20326   545]
 [   96    52  2289]]


Feature Importance

In [20]:
import pandas as pd

feature_importance = pd.DataFrame(
    {"Feature": X_train.columns, "Importance": rf_model.feature_importances_}
)

feature_importance = feature_importance.sort_values(by="Importance", ascending=False)

print(feature_importance.head(20))

                        Feature  Importance
7                         cloud    0.131926
31   humidity_cloud_interaction    0.131180
6                      humidity    0.059509
108           condition_text_42    0.052539
9                 visibility_km    0.046953
28                pm_difference    0.033214
27              temperature_gap    0.027032
30    wind_humidity_interaction    0.023262
17             air_quality_PM10    0.022109
26           day_length_minutes    0.020912
5                   pressure_mb    0.020660
77            condition_text_11    0.019904
16            air_quality_PM2.5    0.019453
8            feels_like_celsius    0.018049
107           condition_text_41    0.016170
0                      latitude    0.015940
11                     gust_kph    0.015758
84            condition_text_18    0.015619
2           temperature_celsius    0.015105
32               heatwave_index    0.014302


In [21]:
condition_cols = [col for col in X.columns if "condition_text" in col]

print(len(condition_cols))
print(condition_cols[:20])

49
['condition_text_1', 'condition_text_2', 'condition_text_3', 'condition_text_4', 'condition_text_5', 'condition_text_6', 'condition_text_7', 'condition_text_8', 'condition_text_9', 'condition_text_10', 'condition_text_11', 'condition_text_12', 'condition_text_13', 'condition_text_14', 'condition_text_15', 'condition_text_16', 'condition_text_17', 'condition_text_18', 'condition_text_19', 'condition_text_20']


In [22]:
d5 = pd.read_csv(
    "../datasets/engineered/D5_feature_engineered.csv"
)

print(
    sorted(
        d5["condition_text"].unique()
    )
)


['Clear', 'Clear ', 'Cloudy', 'Cloudy ', 'Fog', 'Heavy rain', 'Heavy rain at times', 'Heavy snow', 'Light drizzle', 'Light freezing rain', 'Light rain', 'Light rain shower', 'Light sleet', 'Light sleet showers', 'Light snow', 'Light snow showers', 'Mist', 'Moderate or heavy rain in area with thunder', 'Moderate or heavy rain shower', 'Moderate or heavy rain with thunder', 'Moderate or heavy sleet', 'Moderate or heavy snow in area with thunder', 'Moderate or heavy snow showers', 'Moderate or heavy snow with thunder', 'Moderate rain', 'Moderate rain at times', 'Moderate snow', 'Overcast', 'Overcast ', 'Partly Cloudy', 'Partly Cloudy ', 'Partly cloudy', 'Patchy heavy snow', 'Patchy light drizzle', 'Patchy light rain', 'Patchy light rain in area with thunder', 'Patchy light rain with thunder', 'Patchy light snow', 'Patchy light snow in area with thunder', 'Patchy light snow with thunder', 'Patchy moderate snow', 'Patchy rain nearby', 'Patchy rain possible', 'Patchy sleet nearby', 'Patchy s

Remove Condition Text Features

In [23]:
condition_cols = [col for col in X.columns if "condition_text_" in col]

print("Condition Features:", len(condition_cols))

X_no_condition = X.drop(columns=condition_cols)

print(X_no_condition.shape)

Condition Features: 49
(119398, 94)


Train/Test Split Again


In [24]:
from sklearn.model_selection import train_test_split

X_train_nc, X_test_nc, y_train_nc, y_test_nc = train_test_split(
    X_no_condition, y, test_size=0.20, random_state=42, stratify=y
)

print(X_train_nc.shape)
print(X_test_nc.shape)

(95518, 94)
(23880, 94)


Retrain Random Forest

In [25]:
from sklearn.ensemble import RandomForestClassifier

rf_no_condition = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
)

rf_no_condition.fit(X_train_nc, y_train_nc)

print("Random Forest Retrained Successfully")

Random Forest Retrained Successfully


Evaluate Retrained Random Forest

In [26]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred_nc = rf_no_condition.predict(X_test_nc)

print("Accuracy:")
print(round(accuracy_score(y_test_nc, y_pred_nc), 4))

print("\nClassification Report")
print(classification_report(y_test_nc, y_pred_nc))

print("\nConfusion Matrix")
print(confusion_matrix(y_test_nc, y_pred_nc))

Accuracy:
0.9355

Classification Report
              precision    recall  f1-score   support

        High       0.70      0.54      0.61       541
         Low       1.00      0.95      0.97     20902
      Medium       0.63      0.94      0.75      2437

    accuracy                           0.94     23880
   macro avg       0.78      0.81      0.78     23880
weighted avg       0.95      0.94      0.94     23880


Confusion Matrix
[[  291     1   249]
 [   44 19761  1097]
 [   80    69  2288]]


Hyperparameter Tuning

In [27]:
from sklearn.ensemble import RandomForestClassifier

rf_tuned = RandomForestClassifier(
    n_estimators=500,
    max_depth=20,
    min_samples_split=10,
    min_samples_leaf=4,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1,
)

rf_tuned.fit(X_train_nc, y_train_nc)

print("Tuned Random Forest Trained Successfully")

Tuned Random Forest Trained Successfully


Evaluate Tuned Random Forest

In [28]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred_tuned = rf_tuned.predict(X_test_nc)

print("Accuracy:")
print(round(accuracy_score(y_test_nc, y_pred_tuned), 4))

print("\nClassification Report")
print(classification_report(y_test_nc, y_pred_tuned))

print("\nConfusion Matrix")
print(confusion_matrix(y_test_nc, y_pred_tuned))

Accuracy:
0.9347

Classification Report
              precision    recall  f1-score   support

        High       0.64      0.58      0.61       541
         Low       1.00      0.95      0.97     20902
      Medium       0.63      0.91      0.75      2437

    accuracy                           0.93     23880
   macro avg       0.76      0.81      0.78     23880
weighted avg       0.95      0.93      0.94     23880


Confusion Matrix
[[  314     2   225]
 [   52 19783  1067]
 [  121    92  2224]]


Check Overfitting

In [29]:
train_pred_tuned = rf_tuned.predict(X_train_nc)

from sklearn.metrics import accuracy_score

train_acc = accuracy_score(y_train_nc, train_pred_tuned)

print("Train Accuracy:", round(train_acc, 4))

Train Accuracy: 0.9592


Train XGBoost

In [30]:
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_train_nc)

y_test_encoded = label_encoder.transform(y_test_nc)

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1,
)

xgb_model.fit(X_train_nc, y_train_encoded)

print("XGBoost Trained Successfully")

XGBoost Trained Successfully


In [31]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = xgb_model.predict(X_test_nc)

print("Accuracy:", round(accuracy_score(y_test_encoded, y_pred), 4))

print("\nClassification Report")
print(
    classification_report(y_test_encoded, y_pred, target_names=label_encoder.classes_)
)

print("\nConfusion Matrix")
print(confusion_matrix(y_test_encoded, y_pred))

Accuracy: 0.9599

Classification Report
              precision    recall  f1-score   support

        High       0.83      0.50      0.62       541
         Low       0.98      0.98      0.98     20902
      Medium       0.79      0.85      0.82      2437

    accuracy                           0.96     23880
   macro avg       0.87      0.78      0.81     23880
weighted avg       0.96      0.96      0.96     23880


Confusion Matrix
[[  268    28   245]
 [   14 20584   304]
 [   42   324  2071]]


In [32]:
train_pred = xgb_model.predict(
    X_train_nc
)

train_accuracy = accuracy_score(
    y_train_encoded,
    train_pred
)

print(
    "Train Accuracy:",
    round(train_accuracy, 4)
)


Train Accuracy: 0.9917


In [33]:
import joblib

joblib.dump(xgb_model, "../models/xgboost_rainfall_model.pkl")

print("Model Saved Successfully")

Model Saved Successfully


SAVE FEATURE LIST

In [34]:
import joblib

feature_names = X.columns.tolist()

joblib.dump(feature_names, "../models/rainfall_feature_names.pkl")

print("Feature Count:", len(feature_names))
print("Feature Names Saved Successfully")

Feature Count: 143
Feature Names Saved Successfully


In [35]:
print(y.unique())

['Low', 'Medium', 'High']
Categories (3, str): ['Low' < 'Medium' < 'High']


In [36]:
rainfall_class_mapping = {0: "High", 1: "Low", 2: "Medium"}

joblib.dump(rainfall_class_mapping, "../models/rainfall_class_mapping.pkl")

print("Class Mapping Saved")

Class Mapping Saved


In [37]:
import joblib

model = joblib.load("../models/xgboost_rainfall_model.pkl")

print(model.n_features_in_)

94


In [38]:
feature_names = X_no_condition.columns.tolist()

joblib.dump(feature_names, "../models/rainfall_feature_names.pkl")

print(len(feature_names))

94


In [39]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
le.fit(["Low", "Medium", "High"])

print(dict(enumerate(le.classes_)))

{0: np.str_('High'), 1: np.str_('Low'), 2: np.str_('Medium')}


Save Prediction Dataset

In [40]:
df.to_csv("../datasets/processed/climate_rainfall_predictions.csv", index=False)

print("Rainfall Dataset Saved Successfully")

Rainfall Dataset Saved Successfully
